# Project Policy Challenge -- Simple Qwen Starter

**Model:** `Qwen/Qwen2.5-0.5B-Instruct`  
**Task:** For each question, predict `YES`, `NO`, or `INSUFFICIENT_INFO` using frozen PSU policy passages.

This is deliberately a **naive baseline**, not a strong or sophisticated solution. It does just four things:

1. Find three policy snippets using TF-IDF.
2. Send them and the question to Qwen.
3. Read the first decision label from Qwen's response.
4. Calculate accuracy and macro-F1 on the **24 development questions**.

  
The policy passages are fixed, source-attributed **paraphrases** of official PSU pages, not full webpage copies. THESE ARE CLASSROOM RESEARCH MATERIALS, NOT OFFICIAL ADVISING GUIDANCE.

**For Kaggle:** Import this notebook; use **Add Input** to attach `PSU_Qwen_Simple_Starter_Data.zip`. Kaggle may unpack it into a dataset folder automatically. Enable Internet only if needed to install dependencies / download Qwen.


In [ ]:
# 0. Kaggle setup: run this first, before any other import
%pip install -q rank_bm25
%pip uninstall -y -q torchao        # conflicts with peft
%pip install -q -U peft
# If sentence_transformers is missing on this image, uncomment:
# %pip install -q sentence-transformers

In [ ]:
# 1. Check dependencies (no installation)

import torch
import transformers
import sklearn

from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Scikit-learn:", sklearn.__version__)

print("Dependencies loaded successfully!")

PyTorch: 2.14.1+cpu
Transformers: 5.22.1
Scikit-learn: 1.6.1
Dependencies loaded successfully!


In [ ]:
# 2. Load the frozen PSU excerpts, 24 questions, and development labels
from pathlib import Path
import csv, zipfile

filename = 'PSU_24_Questions_v1.csv'
roots = [Path('/kaggle/input'), Path.cwd(), Path('/kaggle/working')]
matches = [p for root in roots if root.exists() for p in root.rglob(filename)
           if (p.parent / 'PSU_24_Dev_Labels_v1.csv').exists()]

# Also works when Kaggle leaves the input ZIP unextracted.
if not matches:
    zips = [p for root in roots if root.exists()
            for p in root.rglob('PSU_Qwen_Simple_Starter_Data.zip')]
    if zips:
        destination = Path('/kaggle/working/psu_simple_data') if Path('/kaggle/working').exists() else Path.cwd()/'psu_simple_data'
        destination.mkdir(exist_ok=True, parents=True)
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(destination)
        matches = list(destination.rglob(filename))
assert matches, 'Attach PSU_Qwen_Simple_Starter_Data.zip using Kaggle Add Input.'
DATA_DIR = matches[0].parent

def read_csv(name):
    with (DATA_DIR/name).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

passages = read_csv('PSU_Policy_Excerpts_v1.csv')
questions = read_csv('PSU_24_Questions_v1.csv')
gold = {r['id']: r['gold'] for r in read_csv('PSU_24_Dev_Labels_v1.csv')}
print(f'{len(passages)} policy passages; {len(questions)} development questions')
print('Example:', questions[0]['question'])


23 policy passages; 24 development questions
Example: A PSU undergraduate has a term GPA of 2.30 and a PSU cumulative GPA of 2.10. Do both numbers satisfy the published GPA thresholds for good academic standing?


In [ ]:
# 3. Very simple retrieval: return the top 3 passages by TF-IDF similarity
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
docs = [p['heading'] + ' ' + p['text'] for p in passages]
doc_vectors = vectorizer.fit_transform(docs)
TOP_K = 3  # Teams can change this.

def retrieve(question):
    q_vector = vectorizer.transform([question])
    scores = (doc_vectors @ q_vector.T).toarray().ravel()
    indices = scores.argsort()[::-1][:TOP_K]
    return [passages[i] for i in indices]

example_evidence = retrieve(questions[0]['question'])
print('Retrieved IDs:', [p['excerpt_id'] for p in example_evidence])


Retrieved IDs: ['S01-E1', 'S01-E2', 'S02-E2']


In [ ]:
# 4. Load Qwen and define a single naive prompting method
import torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu': torch.set_num_threads(4)
print('Loading on', DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE).eval()



Loading on cpu


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
PROMPT = '''Read the PSU policy excerpts and answer the question using ONLY the excerpts.
YES: The excerpts support the proposition in the question.
NO: The excerpts contradict it.
INSUFFICIENT_INFO: The excerpts do not establish either answer.
Do not treat missing evidence as NO.
Output only one label: YES, NO, or INSUFFICIENT_INFO.'''

def predict(question):
    evidence = retrieve(question)
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    messages = [
        {'role': 'system', 'content': PROMPT},
        {'role': 'user', 'content': f'POLICY:\n{context}\n\nQUESTION: {question}'}
    ]
    # print(messages[1]['content'])
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True, truncation=True, max_length=1800
    ).to(DEVICE)
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=20,
                                pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    match = re.match(r'^(?:(?:DECISION|ANSWER)\s*:\s*)?(YES|NO|INSUFFICIENT_INFO)\b', raw, re.I)
    label = match.group(1).upper() if match else 'INVALID'
    return label, raw, [p['excerpt_id'] for p in evidence]

print('Model ready')


Model ready


In [ ]:
# 5. Run evaluation; start with 3, then change to 24
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
import pandas as pd

N_QUESTIONS = 24   # Change to 24 for the whole development set.
results = []
for q in questions[:N_QUESTIONS]:
    prediction, raw, ids = predict(q['question'])
    results.append({'id': q['id'], 'question': q['question'],
                    'gold': gold[q['id']], 'prediction': prediction,
                    'retrieved_ids': ';'.join(ids), 'raw_output': raw})
    print(q['id'], 'gold:', gold[q['id']], 'predicted:', prediction)

truth = [r['gold'] for r in results]
predictions = [r['prediction'] for r in results]
print('\nAccuracy:', round(accuracy_score(truth, predictions), 3))
print('Macro-F1:', round(f1_score(truth, predictions,
    labels=['YES', 'NO', 'INSUFFICIENT_INFO'], average='macro', zero_division=0), 3))
print('Predicted labels:', dict(Counter(predictions)))

# Compare with the deliberately trivial always-YES classifier.
print('Always-YES baseline:', round(sum(label == 'YES' for label in truth) / len(truth), 3))

out = Path('/kaggle/working/PSU_Qwen_Starter_Results.csv') if Path('/kaggle/working').exists() else Path('PSU_Qwen_Starter_Results.csv')
pd.DataFrame(results).to_csv(out, index=False)
print('Saved:', out)


Q01 gold: YES predicted: NO
Q02 gold: NO predicted: NO
Q03 gold: INSUFFICIENT_INFO predicted: NO
Q04 gold: YES predicted: NO
Q05 gold: NO predicted: NO
Q06 gold: INSUFFICIENT_INFO predicted: NO


KeyboardInterrupt: 

## Starting points for teams

This simple baseline is intentionally imperfect. Teams may improve **retrieval**, **prompting**, **decoding**, **data preparation**, **fine-tuning**, or **evaluation**, but should keep the starting checkpoint fixed and compare on the same development questions.

**Important:** The gold labels are development-only and must never be inserted into a model prompt. The hidden test labels will remain private. Do not interpret these 24 examples as live advice about PSU policy.


In [ ]:
# Retrieval recall@1/2/3 for any retriever
import csv
import pandas as pd

# read evidence labels
EV_FILE = 'evidence_labels.csv'
ev_path = next(p for root in [DATA_DIR, Path('/kaggle/input'), Path.cwd()]
               if root.exists() for p in root.rglob(EV_FILE))
with ev_path.open(encoding='utf-8-sig', newline='') as f:
    gold_ev = {r['id']: set(r['gold_evidence'].split(';')) for r in csv.DictReader(f)}

KS = [1, 2, 3]

def eval_recall(retrieve_fn, name):
    """retrieve_fn(question) must return passages ranked best-first (at least 3)."""
    rows = []
    for q in questions:
        gold_ids = gold_ev[q['id']]
        ranked = [p['excerpt_id'] for p in retrieve_fn(q['question'])]
        row = {'id': q['id'],
               'gold_evidence': ';'.join(sorted(gold_ids)),
               'retrieved': ';'.join(ranked[:max(KS)])}
        for k in KS:
            row[f'recall@{k}'] = len(gold_ids & set(ranked[:k])) / len(gold_ids)
        row['missing@3'] = ';'.join(sorted(gold_ids - set(ranked[:3])))
        rows.append(row)
    detail = pd.DataFrame(rows)
    summary = {'retriever': name, **{f'Recall@{k}': round(detail[f'recall@{k}'].mean(), 3) for k in KS}}
    return summary, detail

# current TF-IDF retriever
tfidf_summary, tfidf_detail = eval_recall(retrieve, 'TF-IDF')
print(tfidf_summary)
display(tfidf_detail)

recall_table = [tfidf_summary]   # hybrid results will be appended here later

{'retriever': 'TF-IDF', 'Recall@1': np.float64(0.792), 'Recall@2': np.float64(0.917), 'Recall@3': np.float64(1.0)}


,id,gold_evidence,retrieved,recall@1,recall@2,recall@3,missing@3
0,Q01,S01-E1,S01-E1;S01-E2;S02-E2,1.0,1.0,1.0,
1,Q02,S01-E2,S01-E1;S01-E2;S01-E3,0.0,1.0,1.0,
2,Q03,S01-E3,S01-E3;S01-E2;S04-E2,1.0,1.0,1.0,
3,Q04,S02-E2;S02-E3,S02-E2;S02-E1;S02-E3,0.5,0.5,1.0,
4,Q05,S02-E1;S02-E2,S02-E1;S02-E2;S03-E1,0.5,1.0,1.0,
5,Q06,S02-E3,S02-E3;S02-E2;S01-E1,1.0,1.0,1.0,
6,Q07,S03-E1,S03-E1;S02-E2;S02-E1,1.0,1.0,1.0,
7,Q08,S03-E2,S03-E2;S03-E3;S05-E3,1.0,1.0,1.0,
8,Q09,S04-E3,S04-E3;S02-E2;S02-E3,1.0,1.0,1.0,
9,Q10,S04-E1,S04-E1;S04-E2;S03-E3,1.0,1.0,1.0,


In [ ]:
# 6. Prompt V2: situation vs policy, quote + check conditions + LABEL
PROMPT_V2 = '''You check whether a student's situation satisfies PSU policy rules.
- <policy> contains the ONLY rules you may use as evidence.
- <situation> contains facts about a student and the question to answer.
Treat these facts as true, but they are NOT evidence about what the policy says.
Steps:
1. Find the policy sentence that governs this situation. Quote it with its [ID].
2. List each condition in that rule and check it against the situation.
3. Decide: YES = the policy rule applies and its conditions are met by the situation.
   NO = the policy explicitly says the opposite, or a stated condition is not met.
   INSUFFICIENT_INFO = no policy sentence addresses this situation.
Never quote the <situation> as evidence.
End with: LABEL: YES, NO, or INSUFFICIENT_INFO'''

def parse_label(raw):
    # 优先取 "LABEL:" 后面的值（取最后一个）
    m = re.findall(r'LABEL:\s*(INSUFFICIENT_INFO|YES|NO)\b', raw, re.I)
    if m:
        return m[-1].upper()
    # 兜底：取全文最后出现的标签
    m = re.findall(r'\b(INSUFFICIENT_INFO|YES|NO)\b', raw, re.I)
    return m[-1].upper() if m else 'INVALID'

def predict_v2(question):
    evidence = retrieve(question)
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    messages = [
        {'role': 'system', 'content': PROMPT_V2},
        {'role': 'user', 'content': f'<policy>\n{context}\n</policy>\n\n<situation>\n{question}\n</situation>'}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True, truncation=True, max_length=1800
    ).to(DEVICE)
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=20,
                                pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return parse_label(raw), raw, [p['excerpt_id'] for p in evidence]

print('Prompt V2 ready')

Prompt V2 ready


In [ ]:
label, raw, ids = predict_v2(questions[18]['question'])

print(raw)
print(ids)
print('parsed:', label)

INSUFFICIENT_INFO
['S07-E1', 'S07-E2', 'S06-E1']
parsed: INSUFFICIENT_INFO


In [ ]:
# 7. Run V2 on the development questions
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from collections import Counter
import pandas as pd

N_QUESTIONS = 24
LABELS = ['YES', 'NO', 'INSUFFICIENT_INFO']
results_v2 = []
for q in questions[:N_QUESTIONS]:
    prediction, raw, ids = predict_v2(q['question'])
    results_v2.append({'id': q['id'], 'question': q['question'],
                       'gold': gold[q['id']], 'prediction': prediction,
                       'retrieved_ids': ';'.join(ids), 'raw_output': raw})
    print(q['id'], 'gold:', gold[q['id']], 'predicted:', prediction)

truth_v2 = [r['gold'] for r in results_v2]
pred_v2 = [r['prediction'] for r in results_v2]
print('\nV2 Accuracy:', round(accuracy_score(truth_v2, pred_v2), 3))
print('V2 Macro-F1:', round(f1_score(truth_v2, pred_v2, labels=LABELS,
                                     average='macro', zero_division=0), 3))
print('Predicted labels:', dict(Counter(pred_v2)))

Q01 gold: YES predicted: NO
Q02 gold: NO predicted: NO
Q03 gold: INSUFFICIENT_INFO predicted: NO
Q04 gold: YES predicted: INSUFFICIENT_INFO
Q05 gold: NO predicted: NO
Q06 gold: INSUFFICIENT_INFO predicted: NO
Q07 gold: YES predicted: YES
Q08 gold: NO predicted: INSUFFICIENT_INFO
Q09 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q10 gold: YES predicted: INSUFFICIENT_INFO
Q11 gold: NO predicted: NO
Q12 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q13 gold: YES predicted: INSUFFICIENT_INFO
Q14 gold: NO predicted: NO
Q15 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q16 gold: YES predicted: YES
Q17 gold: NO predicted: NO
Q18 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q19 gold: YES predicted: INSUFFICIENT_INFO
Q20 gold: NO predicted: INSUFFICIENT_INFO
Q21 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q22 gold: YES predicted: INSUFFICIENT_INFO
Q23 gold: NO predicted: INSUFFICIENT_INFO
Q24 gold: INSUFFICIENT_INFO predicted: NO

V2 Accuracy: 0.5
V2 Macro-

In [ ]:
# 8. V3 = V2 prompt + numeric questions decided by code
import re
import torch

# ======================= Part 1: V2 prompt + model =======================
PROMPT_V2 = '''You check whether a student's situation satisfies PSU policy rules.
- <policy> contains the ONLY rules you may use as evidence.
- <situation> contains facts about a student and the question to answer.
Treat these facts as true, but they are NOT evidence about what the policy says.
Steps:
1. Find the policy sentence that governs this situation. Quote it with its [ID].
2. List each condition in that rule and check it against the situation.
3. Decide: YES = the policy rule applies and its conditions are met by the situation.
   NO = the policy explicitly says the opposite, or a stated condition is not met.
   INSUFFICIENT_INFO = no policy sentence addresses this situation.
Never quote the <situation> as evidence.
End with: LABEL: YES, NO, or INSUFFICIENT_INFO'''

def parse_label_v2(raw):
    # prefer the value after "LABEL:" (take the last one)
    m = re.findall(r'LABEL:\s*(INSUFFICIENT_INFO|YES|NO)\b', raw, re.I)
    if m:
        return m[-1].upper()
    # fallback: last label anywhere in the text
    m = re.findall(r'\b(INSUFFICIENT_INFO|YES|NO)\b', raw, re.I)
    return m[-1].upper() if m else 'INVALID'

def model_predict(question, evidence):
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    messages = [
        {'role': 'system', 'content': PROMPT_V2},
        {'role': 'user', 'content': f'<policy>\n{context}\n</policy>\n\n<situation>\n{question}\n</situation>'}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True, truncation=True, max_length=1800
    ).to(DEVICE)
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=20,
                                pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return parse_label_v2(raw), raw

# ======================= Part 2: numeric extraction =======================
NUM_WORDS = {w: str(i) for i, w in enumerate(
    'zero one two three four five six seven eight nine ten eleven twelve thirteen '
    'fourteen fifteen sixteen seventeen eighteen nineteen twenty'.split())}
NUM_RE = re.compile(r'(?<![\w.])\$?(\d+(?:\.\d+)?)(?![\w])%?')      # skips IDs like S01-E1
PRE_OPS = [   # comparison words before a number (longer phrases first)
    (r'greater than or equal to|no less than|at least|a minimum of|minimum of|>=|≥', '>='),
    (r'less than or equal to|no more than|at most|a maximum of|maximum of|up to|<=|≤', '<='),
    (r'more than|greater than|higher than|above|exceeds?|exceeding|>', '>'),
    (r'less than|lower than|fewer than|below|under|<', '<'),
]
POST_OPS = [  # comparison words after a number, e.g. "2.00 or higher"
    (r'or (?:higher|more|above|better|greater)', '>='),
    (r'or (?:lower|less|below|fewer)', '<='),
]
UNITS = ['gpa', 'credit', 'hour', 'day', 'week', 'month', 'year', 'percent', '%']
MODIFIERS = {'cumulative': 'cumulative', 'overall': 'cumulative', 'term': 'term', 'semester': 'term'}
OPS = {'>=': lambda a, b: a >= b, '>': lambda a, b: a > b,
       '<=': lambda a, b: a <= b, '<': lambda a, b: a < b}

def normalize_numbers(text):
    """'six credits' -> '6 credits'"""
    return re.sub(r'\b(' + '|'.join(NUM_WORDS) + r')\b',
                  lambda m: NUM_WORDS[m.group(1).lower()], text, flags=re.I)

def _nearest(before, after, words):
    """keyword closest to the number (text before the number is preferred)"""
    best, dist = None, 10**9
    for w in words:
        for m in re.finditer(re.escape(w), before):
            d = len(before) - m.end()
            if d < dist:
                best, dist = w, d
        m = re.search(re.escape(w), after)
        if m and m.start() + 5 < dist:
            best, dist = w, m.start() + 5
    return best

def tag_number(text, s, e):
    """unit (gpa/credit/...) and modifier (term/cumulative) of a number"""
    before, after = text[max(0, s - 50):s].lower(), text[e:e + 30].lower()
    unit = _nearest(before, after, UNITS)
    mod = MODIFIERS.get(_nearest(before[-45:], after[:15], list(MODIFIERS)))
    return unit, mod

def find_op(text, s, e):
    """comparison operator attached to a policy number, or None if it is not a threshold"""
    before, after = text[max(0, s - 40):s].lower(), text[e:e + 25].lower()
    for pat, op in POST_OPS:
        if re.match(r'\s*(?:' + pat + r')', after):
            return op
    for pat, op in PRE_OPS:
        if re.search(r'(?:' + pat + r')\s*(?:a |an |the |of |gpa |cumulative |term )*\$?$', before):
            return op
    return None

def extract_thresholds(passage):
    text, out = normalize_numbers(passage['text']), []
    for m in NUM_RE.finditer(text):
        op = find_op(text, m.start(), m.end())
        if op:
            unit, mod = tag_number(text, m.start(), m.end())
            out.append((unit, mod, op, float(m.group(1))))
    return out

def extract_student_values(question, evidence):
    text = normalize_numbers(question)
    policy_nums = {float(m.group(1)) for p in evidence
                   for m in NUM_RE.finditer(normalize_numbers(p['text']))}
    out = []
    for m in NUM_RE.finditer(text):
        v = float(m.group(1))
        if v in policy_nums:                       # restated from the policy, not a student value
            continue
        if v.is_integer() and 1900 <= v <= 2100:   # skip years
            continue
        unit, mod = tag_number(text, m.start(), m.end())
        if unit:
            out.append((unit, mod, v))
    return out

def numeric_comparisons(question, evidence):
    """list of (excerpt_id, description, passed)"""
    out = []
    for q_unit, q_mod, q_val in extract_student_values(question, evidence):
        for p in evidence:
            for t_unit, t_mod, op, t_val in extract_thresholds(p):
                if q_unit != t_unit or (q_mod and t_mod and q_mod != t_mod):
                    continue                       # same unit only; never term vs cumulative
                out.append((p['excerpt_id'],
                            f'{q_mod or ""} {q_unit} {q_val:g} {op} {t_val:g}'.strip(),
                            OPS[op](q_val, t_val)))
    return out

# ======================= Part 3: decision =======================
PREDICTION_Q = re.compile(r'\bwill\b|\bdefinitely\b|\bguarantee', re.I)   # asks about a future outcome

def predict_v3(question):
    evidence = retrieve(question)
    ids = [p['excerpt_id'] for p in evidence]
    comps = numeric_comparisons(question, evidence)

    # numeric threshold question -> decided by code
    if comps and not PREDICTION_Q.search(question):
        top_id = next(p['excerpt_id'] for p in evidence if any(c[0] == p['excerpt_id'] for c in comps))
        used = [c for c in comps if c[0] == top_id]
        label = 'YES' if all(ok for _, _, ok in used) else 'NO'
        raw = f'[CODE] {top_id}: ' + '; '.join(f'{d} -> {"TRUE" if ok else "FALSE"}' for _, d, ok in used)
        return label, raw, ids

    # everything else -> V2 model
    label, raw = model_predict(question, evidence)
    return label, raw, ids

# check which dev questions are decided by code
for q in questions:
    ev = retrieve(q['question'])
    if numeric_comparisons(q['question'], ev) and not PREDICTION_Q.search(q['question']):
        print(q['id'], '->', predict_v3(q['question'])[:2])
print('\nV3 ready')

Q01 -> ('YES', '[CODE] S01-E1: term gpa 2.3 >= 2.25 -> TRUE; cumulative gpa 2.1 >= 2 -> TRUE')
Q02 -> ('NO', '[CODE] S01-E1: term gpa 2.3 >= 2.25 -> TRUE; cumulative gpa 1.9 >= 2 -> FALSE')
Q10 -> ('YES', '[CODE] S04-E1: credit 18 <= 45 -> TRUE; credit 18 <= 20 -> TRUE')
Q11 -> ('NO', '[CODE] S04-E1: credit 23 <= 45 -> TRUE; credit 23 <= 20 -> FALSE; credit 23 <= 45 -> TRUE; credit 23 <= 20 -> FALSE')

V3 ready


In [ ]:
# 9. Run V3 on the development questions
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from collections import Counter
import pandas as pd

N_QUESTIONS = 24
LABELS = ['YES', 'NO', 'INSUFFICIENT_INFO']
results_v3 = []
for q in questions[:N_QUESTIONS]:
    prediction, raw, ids = predict_v3(q['question'])
    results_v3.append({'id': q['id'], 'question': q['question'],
                       'gold': gold[q['id']], 'prediction': prediction,
                       'retrieved_ids': ';'.join(ids), 'raw_output': raw})
    print(q['id'], 'gold:', gold[q['id']], 'predicted:', prediction)

truth_v3 = [r['gold'] for r in results_v3]
pred_v3 = [r['prediction'] for r in results_v3]
print('\nV3 Accuracy:', round(accuracy_score(truth_v3, pred_v3), 3))
print('V3 Macro-F1:', round(f1_score(truth_v3, pred_v3, labels=LABELS,
                                     average='macro', zero_division=0), 3))
print('Predicted labels:', dict(Counter(pred_v3)))

Q01 gold: YES predicted: YES
Q02 gold: NO predicted: NO
Q03 gold: INSUFFICIENT_INFO predicted: NO
Q04 gold: YES predicted: INSUFFICIENT_INFO
Q05 gold: NO predicted: NO
Q06 gold: INSUFFICIENT_INFO predicted: NO
Q07 gold: YES predicted: YES
Q08 gold: NO predicted: INSUFFICIENT_INFO
Q09 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q10 gold: YES predicted: YES
Q11 gold: NO predicted: NO
Q12 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q13 gold: YES predicted: INSUFFICIENT_INFO
Q14 gold: NO predicted: NO
Q15 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q16 gold: YES predicted: YES
Q17 gold: NO predicted: NO
Q18 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q19 gold: YES predicted: INSUFFICIENT_INFO
Q20 gold: NO predicted: INSUFFICIENT_INFO
Q21 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q22 gold: YES predicted: INSUFFICIENT_INFO
Q23 gold: NO predicted: INSUFFICIENT_INFO
Q24 gold: INSUFFICIENT_INFO predicted: NO

V3 Accuracy: 0.583
V3 Macro-F1: 0.597
P

In [ ]:
# =====================================================================================
# 10. SFT cells
# =====================================================================================

#  [SFT-1] Setup + load the training file
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)  # confict with peft
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'peft'], check=False)

import json, math, random, re, time
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

LABELS = ['YES', 'NO', 'INSUFFICIENT_INFO']
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()

TRAIN_FILE = 'sft_train_1800.jsonl'
train_path = next(p for root in [Path('/kaggle/input'), Path.cwd(), WORK] if root.exists()
                  for p in root.rglob(TRAIN_FILE))
with train_path.open(encoding='utf-8') as f:
    train_all = [json.loads(line) for line in f]
print(len(train_all), 'training examples from', train_path)
print('labels :', Counter(x['label'] for x in train_all))
print('sources:', Counter(x['source'] for x in train_all))

# Same prompt is used for training and inference.
SFT_SYSTEM_PROMPT = '''Read the policy excerpts and answer the question using ONLY the excerpts.
YES: the excerpts support the proposition in the question.
NO: the excerpts contradict it.
INSUFFICIENT_INFO: the excerpts do not establish either answer (for example, the question asks
what a specific person will decide, an exact personal outcome, or something the excerpts do not cover).
Output only one label: YES, NO, or INSUFFICIENT_INFO.'''

def sft_user_message(question, evidence):
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    return f'POLICY:\n{context}\n\nQUESTION: {question}'

def sft_chat_prompt(question, evidence):
    msgs = [{'role': 'system', 'content': SFT_SYSTEM_PROMPT},
            {'role': 'user', 'content': sft_user_message(question, evidence)}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print('\nExample training prompt:\n' + sft_user_message(train_all[0]['question'], train_all[0]['passages']),
      '\n=>', train_all[0]['label'])


1000 training examples from /content/psu_simple_data/sft_train_1000.jsonl
labels : Counter({'YES': 334, 'NO': 333, 'INSUFFICIENT_INFO': 333})
sources: Counter({'fictional': 700, 'psu_extra': 300})

Example training prompt:
POLICY:
[F24-E2] Cedar Ridge College: Students may request to exceed the 20-hour weekly work limit; the request must be approved by the student employment office.
[F34-E4] At Cedar Ridge College: Credits transferred from other colleges are included in the institutional GPA.
[F26-E2] Cedar Ridge College refunds housing deposits based on cancellation notice: more than 60 days before move-in 100%, 31 to 60 days before move-in 60%, 8 to 30 days before move-in 40%, 7 or fewer days before move-in 10%.

QUESTION: Under the published rules, can a student seek student employment office approval for working more than 20 hours in a week? 
=> YES


In [ ]:

# [SFT-2] Training / inference / evaluation functions
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def sft_encode(ex, max_len=768):
    p_ids = tokenizer(sft_chat_prompt(ex['question'], ex['passages']), add_special_tokens=False)['input_ids']
    t_ids = tokenizer(ex['label'] + '<|im_end|>', add_special_tokens=False)['input_ids']
    p_ids = p_ids[-(max_len - len(t_ids)):]
    return {'input_ids': p_ids + t_ids, 'labels': [-100] * len(p_ids) + t_ids}   # loss only on the answer

def sft_collate(batch):
    L = max(len(b['input_ids']) for b in batch)
    pad = tokenizer.pad_token_id
    ids = torch.tensor([b['input_ids'] + [pad] * (L - len(b['input_ids'])) for b in batch])
    lab = torch.tensor([b['labels'] + [-100] * (L - len(b['labels'])) for b in batch])
    att = torch.tensor([[1] * len(b['input_ids']) + [0] * (L - len(b['input_ids'])) for b in batch])
    return ids, att, lab

def load_base_fp32():
    # fp32 master weights for stable LoRA training; fp16 autocast for speed (T4 has no bf16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE)

def train_lora(train_data, seed=0, epochs=2, lr=2e-4, batch_size=4, grad_accum=2, r=16, alpha=32, log_every=25):
    """LoRA SFT. Memory-saving: logits are computed only at answer positions (Qwen's vocab is 152k,
    full logits for every prompt token would need several GB). Effective batch = batch_size * grad_accum."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    m = get_peft_model(load_base_fp32(), LoraConfig(
        r=r, lora_alpha=alpha, lora_dropout=0.05, task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    m.print_trainable_parameters()
    base = m.get_base_model()          # LoRA layers are already injected inside it
    enc = [sft_encode(x) for x in train_data]
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr, weight_decay=0.0)
    n_batches = math.ceil(len(enc) / batch_size)
    steps = epochs * math.ceil(n_batches / grad_accum)
    warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda')
    m.train(); step, t0, losses = 0, time.time(), []
    for ep in range(epochs):
        order = list(range(len(enc))); random.shuffle(order)
        batches = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
        for b_idx, b in enumerate(batches):
            ids, att, lab = (t.to(DEVICE) for t in sft_collate([enc[j] for j in b]))
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
                hidden = base.model(input_ids=ids, attention_mask=att).last_hidden_state
                tgt = lab[:, 1:]
                keep = tgt != -100                                  # next-token targets inside the answer
                logits = base.lm_head(hidden[:, :-1][keep])
            loss = torch.nn.functional.cross_entropy(logits.float(), tgt[keep])
            scaler.scale(loss / grad_accum).backward()
            losses.append(loss.item())
            if (b_idx + 1) % grad_accum == 0 or b_idx + 1 == len(batches):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
                step += 1
                if step % log_every == 0 or step == steps:
                    print(f'epoch {ep + 1} step {step}/{steps} loss {np.mean(losses[-log_every * grad_accum:]):.4f} '
                          f'({time.time() - t0:.0f}s)')
    m.eval()
    return m

def sft_parse(raw):
    hit = re.search(r'\b(INSUFFICIENT_INFO|YES|NO)\b', raw.upper())
    return hit.group(1) if hit else 'INVALID'

@torch.inference_mode()
def sft_predict(m, question):
    evidence = retrieve(question)
    enc = tokenizer(sft_chat_prompt(question, evidence), return_tensors='pt', add_special_tokens=False).to(DEVICE)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
        out = m.generate(**enc, do_sample=False, max_new_tokens=8, pad_token_id=tokenizer.pad_token_id)
    raw = tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return sft_parse(raw), raw, [p['excerpt_id'] for p in evidence]

def sft_evaluate(m, name, verbose=False):
    m.eval(); rows = []
    for q in questions:
        pred, raw, ids = sft_predict(m, q['question'])
        rows.append(dict(id=q['id'], question=q['question'], gold=gold[q['id']], prediction=pred,
                         evidence_ids=ids[0], retrieved_ids=';'.join(ids), raw_output=raw))
        if verbose:
            print(q['id'], 'gold:', gold[q['id']], 'pred:', pred)
    df = pd.DataFrame(rows)
    res = dict(system=name,
               accuracy=round(accuracy_score(df.gold, df.prediction), 3),
               macro_f1=round(f1_score(df.gold, df.prediction, labels=LABELS, average='macro', zero_division=0), 3),
               pred_counts=dict(Counter(df.prediction)))
    return res, df

print('SFT functions ready')

SFT functions ready


In [ ]:
# [SFT-3] Main system: train on all 1800 examples, evaluate on the 24 dev questions
import gc
if 'model' in globals():
    del model                      # the fp16 baseline model from the starter cells is no longer needed
for _name in ['sft_model', 'frozen']:
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache()
SEED = 0
sft_model = train_lora(train_all, seed=SEED)
main_res, main_df = sft_evaluate(sft_model, f'SFT all 1800 (seed={SEED})', verbose=True)
print(main_res)
print(pd.DataFrame(confusion_matrix(main_df.gold, main_df.prediction, labels=LABELS),
                   index=['gold ' + l for l in LABELS], columns=['pred ' + l for l in LABELS]))

ADAPTER_DIR = WORK / 'qwen_policy_lora'
sft_model.save_pretrained(ADAPTER_DIR)
main_df.to_csv(WORK / 'dev_predictions_sft.csv', index=False)
print('saved adapter ->', ADAPTER_DIR, '| predictions -> dev_predictions_sft.csv')



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


/tmp/ipykernel_30290/2464462393.py:56: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()


epoch 1 step 25/250 loss 0.3861 (26s)
epoch 1 step 50/250 loss 0.2298 (54s)
epoch 1 step 75/250 loss 0.2530 (68s)
epoch 1 step 100/250 loss 0.1461 (86s)
epoch 1 step 125/250 loss 0.1193 (101s)
epoch 2 step 150/250 loss 0.0624 (116s)
epoch 2 step 175/250 loss 0.0466 (130s)
epoch 2 step 200/250 loss 0.0584 (145s)
epoch 2 step 225/250 loss 0.0796 (160s)
epoch 2 step 250/250 loss 0.0240 (175s)
Q01 gold: YES pred: YES
Q02 gold: NO pred: NO
Q03 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q04 gold: YES pred: NO
Q05 gold: NO pred: YES
Q06 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q07 gold: YES pred: YES
Q08 gold: NO pred: NO
Q09 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q10 gold: YES pred: NO
Q11 gold: NO pred: YES
Q12 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q13 gold: YES pred: YES
Q14 gold: NO pred: NO
Q15 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q16 gold: YES pred: YES
Q17 gold: NO pred: NO
Q18 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q19 gold: YES pred:

saved adapter -> /content/qwen_policy_lora | predictions -> dev_predictions_sft.csv


In [ ]:
# [SFT-4] Control: identical prompt + retrieval, LoRA adapter switched off (zero-shot Qwen)
with sft_model.disable_adapter():
    zs_res, zs_df = sft_evaluate(sft_model, 'No SFT (same prompt)')
print(zs_res)

{'system': 'No SFT (same prompt)', 'accuracy': 0.417, 'macro_f1': 0.311, 'pred_counts': {'NO': 22, 'YES': 2}}


In [ ]:
#  [SFT-6] Frozen system + step-by-step explanation (for the test stage and the screencast)
def load_frozen(adapter_dir=WORK / 'qwen_policy_lora'):
    return PeftModel.from_pretrained(load_base_fp32(), adapter_dir).eval()

def explain(m, question):
    print('QUESTION:', question)
    print('RETRIEVED EVIDENCE (TF-IDF top-3):')
    for p in retrieve(question):
        print(f"  [{p['excerpt_id']}] {p['text']}")
    label, raw, ids = sft_predict(m, question)
    print('RAW QWEN OUTPUT:', repr(raw))
    print('PREDICTION:', label, '| SUPPORTING EVIDENCE:', ids[0], '\n')
    return label

frozen = load_frozen()
for q in questions[:3]:
    explain(frozen, q['question'])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

QUESTION: A PSU undergraduate has a term GPA of 2.30 and a PSU cumulative GPA of 2.10. Do both numbers satisfy the published GPA thresholds for good academic standing?
RETRIEVED EVIDENCE (TF-IDF top-3):
  [S01-E1] For undergraduate good academic standing, PSU requires a term GPA of at least 2.25 and a cumulative PSU GPA of at least 2.00.
  [S01-E2] A student on academic warning who raises the term GPA to at least 2.25 but has a cumulative PSU GPA below 2.00 remains on warning. Returning to good standing requires bringing the cumulative PSU GPA to 2.00 or above.
  [S02-E2] When an undergraduate first earns D+, D, D-, or F in a PSU course, repeats that course at PSU using a letter grade (not Pass/No-Pass), and completes the required repeated-course notice, the first low grade may be excluded from the GPA calculation. Both grades stay on the transcript.
RAW QWEN OUTPUT: 'YES'
PREDICTION: YES | SUPPORTING EVIDENCE: S01-E1 

QUESTION: A PSU undergraduate on academic warning earns a term GPA